<a href="https://colab.research.google.com/github/SujaAK/HindiASR/blob/main/5Ksize_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers -q
!pip install snac google-generativeai deepgram-sdk -q
!pip install datasets huggingface_hub soundfile jiwer -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.1 MB/s eta 0:00:00


In [2]:
!pip install torch==2.11.0+cu128 torchaudio==2.11.0+cu128 torchvision==0.26.0+cu128 --index-url https://download.pytorch.org/whl/cu128 -q

In [3]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
from huggingface_hub import login
login()

import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
gemini_model = genai.GenerativeModel("gemini-2.5-flash")

# Quick test
response = gemini_model.generate_content("Say hello in one word.")
print(response.text)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Hi


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from snac import SNAC

print("Loading Veena model...")
model = AutoModelForCausalLM.from_pretrained(
    "maya-research/veena-tts",
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
tokenizer = AutoTokenizer.from_pretrained("maya-research/veena-tts", trust_remote_code=True)

print("Loading SNAC decoder...")
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").eval().cuda()

print("All models loaded successfully!")

Loading Veena model...


config.json:   0%|          | 0.00/839 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/21.0k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

...

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Loading SNAC decoder...


config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/79.5M [00:00<?, ?B/s]

All models loaded successfully!


In [6]:
START_OF_SPEECH_TOKEN = 128257
END_OF_SPEECH_TOKEN = 128258
START_OF_HUMAN_TOKEN = 128259
END_OF_HUMAN_TOKEN = 128260
START_OF_AI_TOKEN = 128261
END_OF_AI_TOKEN = 128262
AUDIO_CODE_BASE_OFFSET = 128266

def generate_speech(text, speaker="kavya", temperature=0.4, top_p=0.9):
    prompt = f"<spk_{speaker}> {text}"
    prompt_tokens = tokenizer.encode(prompt, add_special_tokens=False)

    input_tokens = [
        START_OF_HUMAN_TOKEN, *prompt_tokens, END_OF_HUMAN_TOKEN,
        START_OF_AI_TOKEN, START_OF_SPEECH_TOKEN
    ]
    input_ids = torch.tensor([input_tokens], device=model.device)
    max_tokens = min(int(len(text) * 1.3) * 7 + 21, 700)

    with torch.no_grad():
        output = model.generate(
            input_ids, max_new_tokens=max_tokens, do_sample=True,
            temperature=temperature, top_p=top_p, repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=[END_OF_SPEECH_TOKEN, END_OF_AI_TOKEN]
        )

    generated_ids = output[0][len(input_tokens):].tolist()
    snac_tokens = [t for t in generated_ids if AUDIO_CODE_BASE_OFFSET <= t < (AUDIO_CODE_BASE_OFFSET + 7*4096)]
    if not snac_tokens:
        raise ValueError("No audio tokens generated")
    return decode_snac_tokens(snac_tokens, snac_model)

def decode_snac_tokens(snac_tokens, snac_model):
    if not snac_tokens or len(snac_tokens) % 7 != 0:
        return None
    snac_device = next(snac_model.parameters()).device
    codes_lvl = [[] for _ in range(3)]
    offsets = [AUDIO_CODE_BASE_OFFSET + i * 4096 for i in range(7)]

    for i in range(0, len(snac_tokens), 7):
        codes_lvl[0].append(snac_tokens[i] - offsets[0])
        codes_lvl[1].append(snac_tokens[i+1] - offsets[1])
        codes_lvl[1].append(snac_tokens[i+4] - offsets[4])
        codes_lvl[2].append(snac_tokens[i+2] - offsets[2])
        codes_lvl[2].append(snac_tokens[i+3] - offsets[3])
        codes_lvl[2].append(snac_tokens[i+5] - offsets[5])
        codes_lvl[2].append(snac_tokens[i+6] - offsets[6])

    hierarchical_codes = []
    for lvl in codes_lvl:
        t = torch.tensor(lvl, dtype=torch.int32, device=snac_device).unsqueeze(0)
        hierarchical_codes.append(t)

    with torch.no_grad():
        audio_hat = snac_model.decode(hierarchical_codes)
    return audio_hat.squeeze().clamp(-1, 1).cpu().numpy()

print("Helper functions ready.")

Helper functions ready.


In [7]:
import soundfile as sf
import IPython.display as ipd

text_hindi = "नमस्ते, यह एक परीक्षण वाक्य है"
audio = generate_speech(text_hindi, speaker="kavya")
sf.write("test_veena.wav", audio, 24000)
display(ipd.Audio("test_veena.wav"))

In [8]:
import re
import time
import json
import os

DOMAIN_PROMPTS = {
    "dairy_order": "एक ग्राहक डेयरी की दुकान को फोन पर दूध, दही, पनीर या घी का ऑर्डर दे रहा है।",
    "payment": "एक ग्राहक या दुकानदार पेमेंट, बिल, चालान या क्रेडिट नोट के बारे में बात कर रहा है।",
    "delivery": "डिलीवरी का समय, लोकेशन या टाइमिंग बदलने को लेकर बातचीत हो रही है।",
    "order": "नया ऑर्डर देना, ऑर्डर कैंसिल करना या रिपीट ऑर्डर को लेकर बातचीत हो रही है।",
    "general": "स्टॉक, रेट, क्वालिटी कंप्लेंट या दुकान बंद होने को लेकर सामान्य बातचीत हो रही है।",
}

SYSTEM_INSTRUCTION = (
    "आप एक भारतीय छोटे व्यापारी (दुकानदार) और उसके ग्राहक के बीच होने वाली स्वाभाविक, "
    "बोलचाल की फोन कॉल बातचीत के वाक्य लिखते हैं। हमेशा देवनागरी लिपि में लिखें, "
    "अंग्रेज़ी के शब्द भी देवनागरी में लिखें (जैसे 'पेमेंट', 'डिलीवरी'), कभी भी लैटिन/रोमन लिपि "
    "का उपयोग न करें। संख्याएं हमेशा शब्दों में लिखें, अंकों में नहीं (जैसे 'दस' न कि '10')। "
    "हर वाक्य अलग, प्राकृतिक और विविध होना चाहिए — अलग-अलग नाम, मात्रा, समय और स्थिति का उपयोग करें, "
    "दोहराव से पूरी तरह बचें।"
)

hinglish_markers = ["पेमेंट", "डिलीवरी", "कन्फर्म", "स्टॉक", "क्वालिटी", "कंप्लेंट",
                     "चालान", "क्रेडिट", "कैंसिल", "रिपीट", "अवेलेबल"]

CHECKPOINT_FILE = "gemini_dataset_checkpoint.json"


def is_devanagari_only(text):
    return not re.search(r'[a-zA-Z]', text)


def tag_language_mix(sentence):
    return "hinglish" if any(m in sentence for m in hinglish_markers) else "hindi"


def load_checkpoint():
    """Load existing progress if a checkpoint file exists."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            records = json.load(f)
        print(f"Resuming from checkpoint: {len(records)} sentences already generated.")
        return records
    return []


def save_checkpoint(records):
    """Save current progress to disk."""
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


def generate_batch_in_one_call(domain, count=50, max_retries=4):
    prompt = (
        f"{SYSTEM_INSTRUCTION}\n\n"
        f"विषय: {DOMAIN_PROMPTS[domain]}\n\n"
        f"{count} अलग-अलग, स्वाभाविक वाक्य लिखें। हर वाक्य एक नई लाइन पर लिखें। "
        f"कोई नंबरिंग, बुलेट पॉइंट, या अतिरिक्त टेक्स्ट न जोड़ें — सिर्फ वाक्य।"
    )
    for attempt in range(max_retries):
        try:
            response = gemini_model.generate_content(prompt)
            raw_lines = response.text.strip().split("\n")
            cleaned = []
            for line in raw_lines:
                line = line.strip()
                line = re.sub(r'^\d+[\.\)]\s*', '', line)
                line = line.strip('-* ').strip('"\'""\'\'')
                line = re.sub(r'\d+', '', line).strip()
                if line and len(line.split()) >= 3:
                    cleaned.append(line)
            return cleaned
        except Exception as e:
            wait = 30 * (attempt + 1)
            print(f"  Retry {attempt+1}/{max_retries} for '{domain}': {e}")
            print(f"  Waiting {wait}s...")
            time.sleep(wait)
    return []


def generate_gemini_dataset_at_scale(target_total=5000, per_call_count=50,
                                       delay_between_calls=15, checkpoint_every=5):
    """
    Generates sentences in batches, saving progress to disk every
    `checkpoint_every` API calls. If interrupted (rate limit, crash,
    disconnect), re-running this function picks up where it left off.
    """
    all_records = load_checkpoint()
    seen = set(r["transcript"] for r in all_records)
    domains = list(DOMAIN_PROMPTS.keys())
    call_num = 0

    while len(all_records) < target_total:
        domain = domains[call_num % len(domains)]
        call_num += 1
        print(f"[Call {call_num}] '{domain}' | Total so far: {len(all_records)}/{target_total}")

        sentences = generate_batch_in_one_call(domain, count=per_call_count)
        added = 0
        for sentence in sentences:
            if sentence in seen or not is_devanagari_only(sentence):
                continue
            seen.add(sentence)
            all_records.append({
                "transcript": sentence,
                "domain_tag": domain,
                "language_mix": tag_language_mix(sentence),
                "sentence_length": len(sentence.split()),
                "source": "gemini"
            })
            added += 1
        print(f"  -> Added {added} unique sentences.")

        if call_num % checkpoint_every == 0:
            save_checkpoint(all_records)
            print(f"  [Checkpoint saved: {len(all_records)} sentences]")

        time.sleep(delay_between_calls)

    save_checkpoint(all_records)
    print(f"\nDone. Final checkpoint saved: {len(all_records)} sentences.")
    return all_records[:target_total]


print("Function ready. Checkpoint file:", CHECKPOINT_FILE)
print("If this run gets interrupted (rate limit etc), just re-run this generation")
print("call again later - it will resume from the last saved checkpoint automatically.")

Function ready. Checkpoint file: gemini_dataset_checkpoint.json
If this run gets interrupted (rate limit etc), just re-run this generation
call again later - it will resume from the last saved checkpoint automatically.


In [9]:
gemini_dataset = generate_gemini_dataset_at_scale(
    target_total=5000,
    per_call_count=50,
    delay_between_calls=15,
    checkpoint_every=5
)

[Call 1] 'dairy_order' | Total so far: 0/5000
  -> Added 50 unique sentences.
[Call 2] 'payment' | Total so far: 50/5000
  -> Added 60 unique sentences.
[Call 3] 'delivery' | Total so far: 110/5000
  -> Added 52 unique sentences.
[Call 4] 'order' | Total so far: 162/5000
  -> Added 50 unique sentences.
[Call 5] 'general' | Total so far: 212/5000
  -> Added 50 unique sentences.
  [Checkpoint saved: 262 sentences]
[Call 6] 'dairy_order' | Total so far: 262/5000


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1310.79ms


  -> Added 66 unique sentences.
[Call 7] 'payment' | Total so far: 328/5000
  -> Added 50 unique sentences.
[Call 8] 'delivery' | Total so far: 378/5000
  -> Added 50 unique sentences.
[Call 9] 'order' | Total so far: 428/5000


  Retry 1/4 for 'order': 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 49.017198626s.
  Waiting 30s...


  Retry 2/4 for 'order': 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 18.205214478s.
  Waiting 60s...
  -> Added 53 unique sentences.
[Call 10] 'general' | Total so far: 481/5000


...

  Retry 3/4 for 'dairy_order': 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 3.786266061s.
  Waiting 90s...


  Retry 4/4 for 'dairy_order': 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 32.363276245s.
  Waiting 120s...
  -> Added 0 unique sentences.
[Call 12] 'payment' | Total so far: 531/5000


  Retry 1/4 for 'payment': 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 16.213139627s.
  Waiting 30s...


  Retry 2/4 for 'payment': 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 45.425378642s.
  Waiting 60s...


KeyboardInterrupt: 

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/HindiASR_Dataset"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Saving to: {SAVE_DIR}")

Mounted at /content/drive
Saving to: /content/drive/MyDrive/HindiASR_Dataset


In [11]:
CHECKPOINT_FILE = f"{SAVE_DIR}/gemini_dataset_checkpoint.json"
AUDIO_CHECKPOINT_FILE = f"{SAVE_DIR}/audio_generation_checkpoint.json"
os.makedirs(f"{SAVE_DIR}/generated_audio", exist_ok=True)

In [12]:
import shutil

# Move your existing Gemini checkpoint into the persistent Drive folder
if os.path.exists("gemini_dataset_checkpoint.json"):
    shutil.copy("gemini_dataset_checkpoint.json", f"{SAVE_DIR}/gemini_dataset_checkpoint.json")
    print("Existing 531-sentence checkpoint copied to Drive.")

# Update paths going forward to use Drive
CHECKPOINT_FILE = f"{SAVE_DIR}/gemini_dataset_checkpoint.json"
AUDIO_CHECKPOINT_FILE = f"{SAVE_DIR}/audio_generation_checkpoint.json"
AUDIO_DIR = f"{SAVE_DIR}/generated_audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

print(f"Checkpoint file: {CHECKPOINT_FILE}")
print(f"Audio checkpoint: {AUDIO_CHECKPOINT_FILE}")
print(f"Audio directory: {AUDIO_DIR}")

Existing 531-sentence checkpoint copied to Drive.
Checkpoint file: /content/drive/MyDrive/HindiASR_Dataset/gemini_dataset_checkpoint.json
Audio checkpoint: /content/drive/MyDrive/HindiASR_Dataset/audio_generation_checkpoint.json
Audio directory: /content/drive/MyDrive/HindiASR_Dataset/generated_audio


In [13]:
import os

# Verify the checkpoint file actually exists and is readable from Drive
print("File exists in Drive:", os.path.exists(CHECKPOINT_FILE))
print("File size:", os.path.getsize(CHECKPOINT_FILE), "bytes")

# Actually load it back and confirm sentence count
import json
with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    verify_data = json.load(f)
print("Sentences confirmed in Drive checkpoint:", len(verify_data))
print("Sample entry:", verify_data[0])

File exists in Drive: True
File size: 148093 bytes
Sentences confirmed in Drive checkpoint: 531
Sample entry: {'transcript': 'नमस्ते, क्या यह विजय डेयरी स्टोर है?', 'domain_tag': 'dairy_order', 'language_mix': 'hindi', 'sentence_length': 7, 'source': 'gemini'}
